# EDA danger incendie — Fourcasters

J'explore ici les données **Météo des forêts** utilisées dans le projet.

Le niveau de 1 à 4 correspond au **danger prévu par Météo-France** pour chaque département à J+1 et J+2. Ce n'est pas le nombre de feux réellement observés.

Pour avoir les librairies du notebook :
`uv sync --group analyse`

## 1. Chargement des données

Je charge la table de danger et la table des départements, puis je fais la jointure avec Pandas.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from google.cloud import bigquery

from fourcasters_dbt.configuration import (
    PROJET_GCP,
    DATASET_ANALYSE,
    configurer_google_cloud,
)

configurer_google_cloud()
client = bigquery.Client(project=PROJET_GCP)

danger_id = f"{PROJET_GCP}.{DATASET_ANALYSE}.fact_danger_incendie"
departements_id = f"{PROJET_GCP}.{DATASET_ANALYSE}.dim_departement"

danger = client.list_rows(danger_id).to_dataframe(create_bqstorage_client=False)
departements = client.list_rows(departements_id).to_dataframe(create_bqstorage_client=False)

danger = danger.merge(
    departements,
    on="numero_departement",
    how="left",
)

danger["date_publication"] = pd.to_datetime(danger["date_publication"])
danger["date_prevision"] = pd.to_datetime(danger["date_prevision"])
danger["reference_time"] = pd.to_datetime(danger["reference_time"])

danger = danger.sort_values(
    ["date_publication", "numero_departement", "echeance"]
).reset_index(drop=True)

danger.head()

**Lecture :** une ligne correspond à un niveau de danger pour un département et une échéance, J1 ou J2.

## 2. Vue générale

In [ ]:
print(f"Nombre de lignes : {len(danger):,}")
print(f"Période de publication : {danger['date_publication'].min().date()} → {danger['date_publication'].max().date()}")
print(f"Nombre de départements : {danger['numero_departement'].nunique()}")
print(f"Échéances : {sorted(danger['echeance'].dropna().unique())}")

display(danger.head())
display(danger.dtypes.to_frame("type"))

**Interprétation :** la période est beaucoup plus courte que l'historique météo. Les conclusions sur les territoires doivent donc rester prudentes.

## 3. Qualité des données

In [ ]:
doublons_id = danger["id_danger_incendie"].duplicated().sum()

qualite = pd.DataFrame({
    "valeurs_manquantes": danger.isna().sum(),
    "pourcentage": (danger.isna().mean() * 100).round(2),
}).sort_values("pourcentage", ascending=False)

niveaux_invalides = (~danger["niveau_danger"].between(1, 4)).sum()

print(f"Identifiants en doublon : {doublons_id}")
print(f"Niveaux hors de 1 à 4 : {niveaux_invalides}")
display(qualite.head(12))

nb_departements = danger["numero_departement"].nunique()

couverture = (
    danger.groupby(["reference_time", "echeance"])
    ["numero_departement"]
    .nunique()
    .rename("departements")
    .reset_index()
)

incompletes = couverture[couverture["departements"] != nb_departements]

print(f"Publications / échéances incomplètes : {len(incompletes)}")
display(incompletes.tail(10))

**Interprétation :** je vérifie surtout l'unicité, les valeurs manquantes et la présence de tous les départements à chaque publication.

## 4. Répartition des niveaux de danger

In [ ]:
labels = {
    1: "Faible",
    2: "Modéré",
    3: "Élevé",
    4: "Très élevé",
}

repartition = (
    danger["niveau_danger"]
    .value_counts(normalize=True)
    .reindex([1, 2, 3, 4], fill_value=0)
    .mul(100)
    .round(2)
)

repartition.index = [labels[i] for i in repartition.index]

display(repartition.to_frame("pourcentage"))

repartition.plot(kind="bar", figsize=(7, 4))
plt.title("Répartition des niveaux de danger")
plt.ylabel("Part des observations (%)")
plt.xlabel("")
plt.xticks(rotation=0)
plt.show()

**Interprétation :** les niveaux 1 et 2 sont généralement beaucoup plus fréquents. Le niveau 4 est rare, ce qui explique en partie pourquoi il est difficile à apprendre pour le modèle.

## 5. J1 et J2

In [ ]:
par_echeance = (
    pd.crosstab(
        danger["niveau_danger"],
        danger["echeance"],
        normalize="columns",
    )
    .mul(100)
    .round(2)
    .reindex([1, 2, 3, 4], fill_value=0)
)

par_echeance.index = [labels[i] for i in par_echeance.index]

display(par_echeance)

par_echeance.plot(kind="bar", figsize=(8, 4))
plt.title("Répartition des niveaux à J1 et J2")
plt.ylabel("Part des observations (%)")
plt.xlabel("")
plt.xticks(rotation=0)
plt.show()

**Interprétation :** J1 et J2 ont des profils proches mais pas identiques. Il est utile de garder l'échéance comme variable dans le modèle.

## 6. Évolution pendant la saison

In [ ]:
danger["mois"] = danger["date_prevision"].dt.month

par_mois = (
    danger.groupby(["mois", "echeance"])
    .agg(
        niveau_moyen=("niveau_danger", "mean"),
        pct_danger_eleve=(
            "niveau_danger",
            lambda s: (s >= 3).mean() * 100,
        ),
    )
    .round(2)
    .reset_index()
)

display(par_mois)

for echeance in ["J1", "J2"]:
    subset = par_mois[par_mois["echeance"] == echeance]
    plt.plot(
        subset["mois"],
        subset["pct_danger_eleve"],
        marker="o",
        label=echeance,
    )

plt.title("Part des niveaux 3 et 4 selon le mois")
plt.xlabel("Mois")
plt.ylabel("Danger élevé ou très élevé (%)")
plt.legend()
plt.show()

**Interprétation :** cette vue montre à quel moment de la saison les niveaux 3 et 4 sont les plus présents. Elle peut ensuite être rapprochée de la saisonnalité météo.

## 7. Départements et régions les plus concernés

In [ ]:
par_departement = (
    danger.groupby(["numero_departement", "departement"])
    .agg(
        observations=("niveau_danger", "size"),
        niveau_moyen=("niveau_danger", "mean"),
        pct_danger_eleve=(
            "niveau_danger",
            lambda s: (s >= 3).mean() * 100,
        ),
        nb_tres_eleve=(
            "niveau_danger",
            lambda s: (s == 4).sum(),
        ),
    )
    .round(2)
    .reset_index()
    .sort_values("pct_danger_eleve", ascending=False)
)

display(par_departement.head(15))

top_departements = par_departement.head(15).sort_values("pct_danger_eleve")
top_departements.plot(
    x="departement",
    y="pct_danger_eleve",
    kind="barh",
    figsize=(9, 6),
    legend=False,
)
plt.title("Départements avec le plus de niveaux 3 ou 4")
plt.xlabel("Part des observations (%)")
plt.ylabel("")
plt.show()

par_region = (
    danger.groupby("region")["niveau_danger"]
    .apply(lambda s: (s >= 3).mean() * 100)
    .sort_values(ascending=False)
    .round(2)
)

display(par_region.to_frame("pct_danger_eleve"))

**Interprétation :** certains territoires reçoivent plus souvent des niveaux élevés sur la période disponible. Comme l'historique est court, je parle surtout de la saison observée et pas d'un classement climatique définitif.

## 8. Journées où le danger est le plus généralisé

In [ ]:
jours_forts = (
    danger.groupby(["date_prevision", "echeance"])
    .agg(
        departements_niveau_3_ou_4=(
            "niveau_danger",
            lambda s: (s >= 3).sum(),
        ),
        departements_niveau_4=(
            "niveau_danger",
            lambda s: (s == 4).sum(),
        ),
        niveau_moyen=("niveau_danger", "mean"),
    )
    .round(2)
    .reset_index()
    .sort_values(
        ["departements_niveau_3_ou_4", "departements_niveau_4"],
        ascending=False,
    )
)

display(jours_forts.head(15))

**Interprétation :** ces dates correspondent aux épisodes où le danger est le plus étendu en France. Elles sont intéressantes à comparer ensuite avec la météo de la même période.

## 9. Différence entre J1 et J2 pour une même publication

In [ ]:
comparaison = (
    danger.pivot_table(
        index=["reference_time", "numero_departement"],
        columns="echeance",
        values="niveau_danger",
        aggfunc="first",
    )
    .dropna(subset=["J1", "J2"])
    .reset_index()
)

comparaison["evolution"] = "Stable"
comparaison.loc[comparaison["J2"] > comparaison["J1"], "evolution"] = "Hausse"
comparaison.loc[comparaison["J2"] < comparaison["J1"], "evolution"] = "Baisse"

variations = (
    comparaison["evolution"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display(variations.to_frame("pourcentage"))

variations.plot(kind="bar", figsize=(7, 4))
plt.title("Différence entre J1 et J2")
plt.ylabel("Part des observations (%)")
plt.xlabel("")
plt.xticks(rotation=0)
plt.show()

**Interprétation :** cette comparaison montre si le niveau prévu à J2 est souvent plus haut, plus bas ou identique à J1 pour une même publication. Ce n'est pas une mesure de précision : J1 et J2 ne concernent pas la même date.

## 10. Lecture rapide

In [ ]:
niveau_dominant = danger["niveau_danger"].mode().iloc[0]
pct_eleve = (danger["niveau_danger"].ge(3).mean() * 100)
dept_plus_concerne = par_departement.iloc[0]
variation_dominante = variations.idxmax()

print("Quelques constats :")
print(f"- Niveau le plus fréquent : {labels[niveau_dominant]}")
print(f"- Part des niveaux 3 ou 4 : {pct_eleve:.1f} %")
print(
    f"- Département le plus souvent en niveau 3 ou 4 sur la période : "
    f"{dept_plus_concerne['departement']} "
    f"({dept_plus_concerne['pct_danger_eleve']:.1f} %)"
)
print(f"- Situation la plus fréquente entre J1 et J2 : {variation_dominante}")

## Bilan

Cet EDA permet de voir :
- si les publications sont complètes ;
- quels niveaux sont les plus fréquents ;
- quand le danger augmente pendant la saison ;
- quels territoires sont les plus concernés ;
- comment J1 et J2 se comparent.

Le point important est de rester prudent : Météo-France fournit un **niveau de danger**, pas le nombre réel de départs de feu.